# 03 — Working Capital Quality Analysis

## Purpose
Assess whether working capital is supported by real cash versus debtors/inventory, and whether debt is funding WC deficits or genuine growth.

**Bank context:** A borrower may show positive working capital on paper, but if it's entirely driven by slow-moving debtors or aging inventory, the quality is poor. Banks want to see **cash-backed WC**.

**Assessment flow (from AU SME Borrower Model):**
1. Remove cash from current assets
2. Test the raw WC gap (CA excl cash - CL)
3. Add cash back to test true short-term liquidity
4. Check if borrowing is needed to make WC positive
5. Analyse composition (cash share, debtor share, inventory share)

---

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from src.data_generation import generate_sme_dataset
from src.working_capital import borrower_wc_analysis, portfolio_wc_flags, analyse_working_capital

pd.set_option('display.float_format', '{:,.2f}'.format)
sns.set_theme(style='whitegrid', palette='muted')

df = generate_sme_dataset(n_borrowers=80, seed=42)
print(f'Loaded {df["borrower_id"].nunique()} borrowers')

## Base Case Borrower — Working Capital Waterfall

Step-by-step WC analysis for Base Case AU SME Pty Ltd.

In [ ]:
wc_table = borrower_wc_analysis(df, borrower_id=0)

print('='*70)
print('WORKING CAPITAL ANALYSIS — Base Case AU SME Pty Ltd')
print('='*70)
display(wc_table)

In [ ]:
# Waterfall chart for FY0
fy0_row = df[(df['borrower_id']==0) & (df['period']=='FY0')].iloc[0]
wc = analyse_working_capital(fy0_row)

labels = ['CA excl Cash', '- CL', '= Raw WC\n(excl Cash)', '+ Cash', '= Final WC']
values = [
    wc['ca_excl_cash'],
    -wc['current_liabilities'],
    wc['raw_wc_excl_cash'],
    wc['cash'],
    wc['final_wc_incl_cash'],
]

fig, ax = plt.subplots(figsize=(12, 6))
colors = ['#2196F3', '#F44336', '#FF9800' if values[2] < 0 else '#4CAF50', '#2196F3', '#4CAF50']

# Simple bar chart showing the waterfall components
bars = ax.bar(labels, [v / 1e6 for v in values], color=colors, alpha=0.8, edgecolor='white', linewidth=1.5)

for bar, v in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'${v/1e6:.1f}M', ha='center', fontsize=11, fontweight='bold')

ax.axhline(0, color='black', linewidth=0.8)
ax.set_title(f'Working Capital Waterfall — FY0 (Flag: {wc["flag"]})', fontsize=14, fontweight='bold')
ax.set_ylabel('A$ Millions')

plt.tight_layout()
plt.savefig('../outputs/figures/03_wc_waterfall.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nConclusion: {wc["flag"]} — {wc["comment"]}')
print(f'Cash share of CA: {wc["cash_share_of_ca"]:.0%}')
print(f'Debtor share of CA: {wc["debtor_share_of_ca"]:.0%}')
print(f'Debt funding WC? {wc["debt_funding_wc"]}')

## Current Asset Composition — Pie Chart

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

components = ['Cash', 'Debtors', 'Inventory', 'Other CA']
sizes = [wc['cash'], wc['debtors'], wc['inventory'], wc['other_ca']]
colors_pie = ['#4CAF50', '#FF9800', '#2196F3', '#9E9E9E']

wedges, texts, autotexts = ax.pie(
    sizes, labels=components, autopct='%1.0f%%',
    colors=colors_pie, startangle=90, textprops={'fontsize': 12}
)
ax.set_title('Current Asset Composition — FY0', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/figures/03_ca_composition.png', dpi=150, bbox_inches='tight')
plt.show()

## Portfolio Working Capital Flags

In [ ]:
wc_flags = portfolio_wc_flags(df)

flag_counts = wc_flags['flag'].value_counts()
print('Working Capital Flags (FY0):')
for flag in ['GREEN', 'AMBER', 'RED']:
    count = flag_counts.get(flag, 0)
    pct = count / len(wc_flags) * 100
    print(f'  {flag}: {count} ({pct:.0f}%)')

fig, ax = plt.subplots(figsize=(8, 5))
flag_colors = {'GREEN': '#4CAF50', 'AMBER': '#FF9800', 'RED': '#F44336'}
for flag in ['GREEN', 'AMBER', 'RED']:
    if flag in flag_counts.index:
        ax.bar(flag, flag_counts[flag], color=flag_colors[flag], alpha=0.8)
ax.set_title('Portfolio Working Capital Quality Distribution', fontweight='bold')
ax.set_ylabel('Number of Borrowers')

plt.tight_layout()
plt.savefig('../outputs/figures/03_wc_portfolio.png', dpi=150, bbox_inches='tight')
plt.show()

---

## Key Takeaways

1. **Cash-backed WC is preferred** — if WC is only positive because of debtors/inventory, the quality is weaker
2. **Borrowing to fund WC is a warning sign** — debt should fund growth, not plug short-term liquidity gaps
3. **The assessment flow (remove cash → test gap → add cash back)** reveals the underlying WC quality
4. **Debtor concentration** — high receivables share can make WC look better than it really is

**Next:** [04_Credit_Scoring_and_PD_Estimation.ipynb](04_Credit_Scoring_and_PD_Estimation.ipynb)